# Module B04 — Repeating Work: Loops

## Exercise 6: Loops that never stop

An error stops your program, prints a traceback, and names a line. An infinite
loop does none of those things. It sits there. The cell shows a running marker,
nothing appears, and the only information you get is that something is wrong
somewhere.

Of the two, the error is the kinder outcome by a wide margin.

This notebook shows you four ways to write a loop that never ends, so that you
recognise each of them on sight. None of them will hang this notebook, because
every one of them runs under a safety cap. Section 2 explains what that cap is
and why it does not belong in real code.

| | |
|---|---|
| Time | About 40 minutes |
| You need | This notebook |
| Comes after | Exercise 5, enumerate and zip |

---

## 1. What "never terminates" means, and how to stop one

A `while` loop ends when its condition becomes `False`. If nothing in the body
can ever make that happen, the loop runs until something outside it intervenes.

You will write one. Everybody does. So learn the recovery before the failure:

- **In Jupyter or VS Code:** press the stop button on the toolbar, or press `i`
  twice with the notebook focused. This is called interrupting the kernel. Your
  variables survive.
- **In a terminal:** press `Ctrl-C`.

Either way you get a `KeyboardInterrupt`, which is Python telling you that you
stopped it rather than that it stopped itself.

One loop is worse than a plain infinite loop, and that is an infinite loop with
`print` in it, because it also fills your notebook with output until the browser
struggles. If you find yourself writing a long-running loop, get the stopping
condition right before you add the printing.

---

## 2. The safety cap, which is scaffolding

Every broken loop in this notebook is wrapped in a counter that forces it to
stop.

```python
guard = 0
while some_broken_condition:
    guard += 1
    if guard > 20:
        print("stopped by the safety cap")
        break
    ...
```

**This exists so that a teaching notebook cannot freeze your kernel. It is not a
technique.**

A cap in code you ship does not fix the loop. It hides it. The loop is still
wrong, it still fails to do the job, and now it fails quietly after twenty
passes instead of loudly forever, which is harder to notice and harder to
diagnose. Somebody reading that code in a year cannot tell whether twenty is a
real limit or a bandage.

Where a limit is genuinely part of the specification, say so and name it:
`max_attempts = 3` in exercise 3 is a rule of the game, not a guard against a
bug you did not find. The difference is whether you could explain the number to
the person who asked for the feature.

---

## 3. Cause one: the variable never changes

Part three of the three parts is missing. The condition asks about `n`, and
nothing in the body touches `n`.

In [ ]:
n = 5
guard = 0

while n > 0:
    guard += 1
    if guard > 8:
        print("stopped by the safety cap, with n still", n)
        break
    print("n is", n)

print("the loop never reduced n")

`n is 5` eight times. Without the cap this prints `n is 5` until you stop it.

This is the most common infinite loop and it usually arrives by deletion: the
line was there, something got rearranged, and the line that changed `n` ended up
somewhere else or got removed with a block that was cut.

The reading habit that catches it: for every `while`, find the name in the
condition, then find the line in the body that changes that name. If you cannot
point at that line, the loop does not stop.

---

## 4. Cause two: it changes in the wrong direction

The variable changes on every pass. It moves away from the condition instead of
towards it.

In [ ]:
n = 5
guard = 0

while n > 0:
    guard += 1
    if guard > 8:
        print("stopped by the safety cap, with n now", n)
        break
    print("n is", n)
    n = n + 1

print("n went up, and the condition asked it to come down")

`5, 6, 7, 8...` and the condition `n > 0` gets more true rather than less.

`+` where you meant `-` is one keystroke, and it produces a loop that is harder
to read past than a missing line, because there **is** a line that changes `n`
and it looks like the loop is doing its job.

Check the direction as well as the presence. The condition and the change have
to agree: `n > 0` wants `n` going down, `n < limit` wants `n` going up.

---

## 5. Cause three: a condition that can only be missed

The variable changes, in the right direction, and the condition is still never
`False`, because it tests for one exact value and the loop steps straight past
it.

In [ ]:
n = 0
guard = 0

while n != 10:
    guard += 1
    if guard > 8:
        print("stopped by the safety cap, with n at", n)
        break
    print("n is", n)
    n += 3

print("0, 3, 6, 9, 12 ... 10 is never one of them")

`0, 3, 6, 9, 12, 15, 18, 21`. It jumps from 9 to 12 and 10 never happens, so
`n != 10` is `True` forever.

**Prefer `<` and `>` over `!=` and `==` in a loop condition on numbers that
step.** `while n < 10` stops whether the step lands on 10 or steps over it.
`while n != 10` needs the step to hit exactly, and it is one changed step size
away from running forever.

This is the cause that survives testing, because it works perfectly with a step
of 1 and breaks the day somebody changes the step to 3.

---

## 6. The same cause, with floats

Module B02 established that floats are approximate. Put an approximate value in
an exact condition and you have cause three again, with no visible step size to
blame.

In [ ]:
total = 0.0
guard = 0

while total != 1.0:
    guard += 1
    if guard > 15:
        print("stopped by the safety cap, with total at", total)
        break
    total += 0.1

print("adding 0.1 ten times does not produce exactly 1.0")

The cap caught it well past 1.0. Adding `0.1` ten times gives
`0.9999999999999999`, which is not `1.0`, so the loop sailed through the value
it was waiting for.

Same rule, and it is not optional here: use `<` or `>` with floats, or compare
with a tolerance. Never `!=` and never `==`.

---

## 7. Cause one in disguise: `continue` above the change

Exercise 3 said to put the line that changes your variable at the top of a
`while` body. Here is what happens when it goes at the bottom instead and a
`continue` sits above it.

In [ ]:
n = 0
guard = 0

while n < 5:
    guard += 1
    if guard > 8:
        print("stopped by the safety cap, with n stuck at", n)
        break
    if n == 2:
        continue          # skips the rest of the body, including n += 1
    print("n is", n)
    n += 1

print("n reached 2 and stayed there")

`0, 1`, and then silence.

`continue` goes back to the top of the loop without running the rest of the
body, and the rest of the body is where `n += 1` lives. From the pass where
`n == 2`, nothing changes `n` ever again.

This is cause one, hiding. The line that changes `n` exists, you can point at
it, and on some passes it does not run. The fix is the habit from exercise 3:
change the variable at the top, above every branch.

---

## 8. A condition that crashes instead, run on purpose

Not every broken condition hangs. Some raise, and when they do you have been let
off lightly.

In [ ]:
typed = "10"          # what input() hands you: text, always
n = 0

while n < typed:
    n += 1

```
TypeError: '<' not supported between instances of 'int' and 'str'
```

**`TypeError`** means the types were wrong for the operation. `n` is a whole
number, `typed` is text, and Python refuses to guess what "less than" means
between the two.

Some languages would compare them anyway, by a rule you did not know about, and
give you a loop that runs a strange number of times and never explains itself.
Python stopping here is the feature.

The fix is the conversion you meant, and exercise 3 gave you the check to put in
front of it.

In [ ]:
typed = "10"
limit = int(typed)
n = 0

while n < limit:
    n += 1

print("finished with n =", n)

---

## 9. Why `for` over a `range` cannot do this

A `for` loop over a `range` has its number of passes decided before the first
pass runs. Nothing in the body can change it, including reassigning the loop
name.

In [ ]:
for i in range(5):
    i = 100          # this does not affect the loop at all
    print("i is", i)

print("still exactly five passes")

Five passes, whatever the body does to `i`.

This is the strongest argument for reaching for `for` whenever the number of
passes is knowable. It removes an entire category of failure rather than
requiring you to avoid it.

There is one `for` loop that can run forever, and you met the ingredients in
exercise 5: a `for` over a list you keep adding to.

In [ ]:
items = [1, 2, 3]
guard = 0

for item in items:
    guard += 1
    if guard > 10:
        print("stopped by the safety cap")
        break
    items.append(item)

print("the list grew to", len(items), "items and the loop kept finding more")

The loop asks the list for the next item each pass, and each pass put another
item on the end. Exercise 5 said not to change a list you are looping over
because it silently skips items. This is the other half of that rule: adding
instead of removing does not skip anything, it never finishes.

---

# Your turn

**Do not delete the `# ANSWER n` marker lines.** The self-check uses them.

Every task below keeps its safety cap while you work, so a wrong fix stops
rather than hangs. When a loop is right, its cap never fires. Leave the caps in
place here; take them out of anything you write elsewhere.

### Task 1

Predict, **before running** anything. For each loop write `ends` or
`never ends`, and a few words on why.

In [ ]:
# ANSWER 1
# (a) n = 0    while n < 5:   n += 1                    ___
# (b) n = 0    while n < 5:   print(n)                  ___
# (c) n = 10   while n > 0:   n += 1                    ___
# (d) n = 0    while n != 7:  n += 2                    ___
# (e) n = 0    while n < 5:   n += 1  (n reset to 0 first line of body)   ___
# (f) for i in range(3):      i = 0                     ___

# The two safe ones, to check yourself against:
n = 0
while n < 5:
    n += 1
print("(a) finished with n =", n)

for i in range(3):
    i = 0
    print("(f) pass, i is", i)

### Task 2

Loop A is cause one: nothing changes the variable. Fix it.

The cap must never fire once your fix is right.

In [ ]:
# ANSWER 2
n = 5
guard = 0

while n > 0:
    guard += 1
    if guard > 50:
        print("the safety cap fired, the loop is still broken")
        break
    print("n is", n)
    ___

print("finished with n =", n)

### Task 3

Loop B is cause two: it changes in the wrong direction. Fix the change so the
loop counts down to zero.

In [ ]:
# ANSWER 3
n = 5
guard = 0

while n > 0:
    guard += 1
    if guard > 50:
        print("the safety cap fired, the loop is still broken")
        break
    print("n is", n)
    n = ___

print("finished with n =", n)

### Task 4

Loop C is cause three: the condition tests for one exact value and the step goes
past it. Fix the **condition**, and leave the step of 3 alone.

Then say in the comment what would have gone wrong if you had changed the step
to 1 and kept the condition.

In [ ]:
# ANSWER 4
n = 0
guard = 0

while ___:
    guard += 1
    if guard > 50:
        print("the safety cap fired, the loop is still broken")
        break
    print("n is", n)
    n += 3

print("finished with n =", n)

# Why is changing the step to 1 a worse fix than changing the condition? ___

### Task 5

You are reading a colleague's code and you find this in a program that has been
running in production for a year:

```python
guard = 0
while not finished:
    guard += 1
    if guard > 1000:
        break
    ...
```

Say what you would do about it, and what you would need to find out first. Then
write down the one question that decides whether that `1000` stays.

There is no single correct answer. The reasoning is the exercise.

In [ ]:
# ANSWER 5
what_i_would_do = "___"
what_i_would_need_to_find_out = "___"
the_question_that_decides_it = "___"

---

## Self-check

You do not need to understand this cell. It is machinery, not material.

In [ ]:
def _answer(marker):
    """Find the most recent cell you ran that contains the given marker."""
    try:
        matches = [c for c in _ih if marker in c and "def _answer" not in c]
    except NameError:
        print("Run this in Jupyter or VS Code so the self-check can see your cells.")
        return ""
    return matches[-1] if matches else ""


def check(passed, message):
    print(("PASS  " if passed else "FAIL  ") + message)
    return bool(passed)

NEXT = "That is module B04 finished. Module B05 makes lists precise."

a1, a2, a3, a4, a5 = (_answer("# ANSWER 1"), _answer("# ANSWER 2"),
                      _answer("# ANSWER 3"), _answer("# ANSWER 4"),
                      _answer("# ANSWER 5"))

results = [
    check(a1.count("___") == 0 and "never" in a1.lower(),
          "Task 1: all six loops judged before running, at least one never ending"),
    check(a2.count("___") == 0 and ("n -= 1" in a2 or "n = n - 1" in a2
          or "n-=1" in a2.replace(" ", "")),
          "Task 2: loop A now reduces n inside the body"),
    check("the safety cap fired" in a2, "Task 2: the cap is still in place"),
    check("n = n - 1" in a3 or "n-1" in a3.replace(" ", ""),
          "Task 3: loop B counts down instead of up"),
    check("!=" not in a4.split("while")[-1].split(":")[0],
          "Task 4: the condition no longer tests for one exact value"),
    check("<" in a4.split("while")[-1].split(":")[0]
          or ">" in a4.split("while")[-1].split(":")[0],
          "Task 4: the condition uses a comparison that a step can pass"),
    check("worse fix than changing the condition? ___" not in a4,
          "Task 4: you said why changing the step is the worse fix"),
    check(a5.count("___") == 0 and "the_question_that_decides_it" in a5,
          "Task 5: all three answers written"),
]

print()
failed = results.count(False)
print("%d of %d checks failing. Keep going." % (failed, len(results)) if failed
      else "All %d checks passing. %s" % (len(results), NEXT))

---

## What you learned

- A loop that never terminates gives you no traceback and no line number. Stop
  it by interrupting the kernel, or `Ctrl-C` in a terminal.
- The safety cap in this notebook is scaffolding so nothing here can hang. A cap
  in shipped code hides the bug instead of removing it.
- Cause one: nothing in the body changes the name in the condition.
- Cause two: it changes in the wrong direction, which is harder to see because a
  change is visibly there.
- Cause three: the condition tests for one exact value and the step goes past
  it. Prefer `<` and `>` over `!=` on numbers that step.
- Floats make cause three worse, because the value you are waiting for may never
  be produced exactly.
- `continue` above the line that changes your variable turns a working loop into
  cause one. Change the variable at the top of the body.
- Comparing a number with text raises `TypeError` rather than looping strangely,
  which is Python doing you a favour.
- A `for` over a `range` cannot run forever. A `for` over a list you keep
  appending to can.

## Before you move on

- [ ] You can name the three parts of a `while` and which cause matches each
      missing part.
- [ ] You know how to interrupt a running cell in the editor you use.
- [ ] You can say why a safety cap does not belong in code you ship.
- [ ] You fixed a loop by changing its condition rather than its step, and can
      say why that was the better fix.

**Next:** module B05, where lists stop being the small thing you were told
enough about to get through these loops, and become the subject.